# Hyperparameter Optimization

In [1]:
import seaborn as sns
import warnings
import joblib
from pathlib import Path

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
import optuna.visualization as vis

from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical

import sys 
sys.path.append('..')  

from module.dataload import DPN_data
import ymlconfig

%matplotlib inline
%load_ext autoreload
%autoreload 2

warnings.filterwarnings('ignore')
np.set_printoptions(precision=3)  # decimal places for outputs from numpy
pd.set_option("display.precision", 3)  # decimal places for outputs from pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

c:\Users\acbriza\anaconda3\envs\dpncf\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load / Reload Selection Utility Functions

In [ ]:
from utils2.optimization import *

----

## Read Config File

In [17]:
config_path = Path(r'experiments\binary')
config_file = config_path / "optimization_config_dev.yml"
#config_file = config_path / "optimization_config_final.yml"
config_dict = ymlconfig.load_config(config_file)
config = ymlconfig.dict_to_namespace(config_dict)
config_dict

{'experiment': {'summary': 'binary classification - hyperparameter optimization(final experiment)',
  'classification_type': 'binary',
  'stage': 'hyperparameter_optimization',
  'tag': 'development',
  'verbosity': 1,
  'random_seed': 42},
 'data': {'dataset_path': '../dataset/Sudoscan Working File with Stats.xlsx'},
 'model': {'name': 'catboost'},
 'param_space': {'iterations': {'min': 100, 'max': 500},
  'depth': {'min': 4, 'max': 10},
  'learning_rate': {'min': 0.01, 'max': 0.1},
  'l2_leaf_reg': {'min': 1, 'max': 9}},
 'optimization': {'scoring': 'roc_auc',
  'k_splits_outer': 3,
  'n_repeats_outer': 2,
  'k_splits_inner': 3,
  'n_iter': 5},
 'evaluation': {'confidence': 0.95,
  'k_splits_outer': 3,
  'n_repeats_outer': 2,
  'k_splits_inner': 3,
  'n_iter': 5}}

## Data Loading

In [4]:
D = DPN_data(config.data.dataset_path)
D.load(classification=config.experiment.classification_type)

dfdpn = D.df
data_cols = dfdpn.drop(D.non_data_cols, axis=1, errors="ignore").columns
X = dfdpn[data_cols]
y = dfdpn['Confirmed_Binary_DPN']
X.shape, y.shape

((190, 40), (190,))

## CatBoost BayesSearchCV Optimization  

----

In [11]:
from utils2.optimization import *

### Param Space Definition

In [ ]:
# this one works, but actually performs like grid search
param_space = {
    'iterations': [10], #[100, 200, 500],
    'depth': [4, 6], # [4, 6, 8, 10],
    'learning_rate': [0.01, 0.05], #[0.01, 0.05, 0.1],
    'l2_leaf_reg': [1, 3], # [1, 3, 5, 7, 9]
}

# correct implementation but not tried
from skopt.space import Integer, Real
param_space = {
    'iterations': Integer(100, 500),
    'depth': Integer(4, 10),
    'learning_rate': Real(0.01, 0.1, prior='log-uniform'),
    'l2_leaf_reg': Integer(1, 9),
}


### Optimization 

In [ ]:
model = CatBoostClassifier(
    verbose=0,  # Show progress every 100 iterations
    loss_function="Logloss",
    eval_metric="AUC",
    random_state=42,
    thread_count=-1  # Use all CPU cores
)

results = nested_cv_youden(
    X=X.values,
    y=y.values,
    model=model,
    param_space=param_space,
    n_splits_outer=5,
    n_repeats_outer=5,
    n_splits_inner=3,
    n_iter=30
)

results

### Calculate Confidence Interval 

In [ ]:
import numpy as np

def mean_confidence_interval(results, metric, confidence=0.95):
    scores = [fold[metric] for fold in results['folds']]
    
    scores = np.array(scores)
    n = len(scores)
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    stderr = std / np.sqrt(n)

    z = 1.96  # 95% normal approx
    margin = z * stderr

    return {
        "mean": mean,
        "std": std,
        "ci_lower": mean - margin,
        "ci_upper": mean + margin,
        "n_folds": n
    }

In [ ]:
youden_ci = mean_confidence_interval(results, "youden")
roc_ci = mean_confidence_interval(results, "roc_auc")

print("Youden 95% CI:", youden_ci)
print("ROC-AUC 95% CI:", roc_ci)

### Train Final Model

#### Define Function for Retraining

In [ ]:
from sklearn.model_selection import StratifiedKFold
from skopt import BayesSearchCV
import numpy as np
from sklearn.metrics import roc_curve

def train_final_model(X, y, model, param_space, n_splits_inner=3, n_iter=30, random_state=42, n_jobs=-1):
    """
    Train the final deployable model on ALL data:
    1. BayesSearchCV to find best hyperparameters (inner CV on full dataset)
    2. Refit on full dataset with best params
    3. Threshold via Youden index on OOF predictions
    """

    inner_cv = StratifiedKFold(n_splits=n_splits_inner, shuffle=True, random_state=random_state)

    # Step 1: Hyperparameter search on full data
    opt = BayesSearchCV(
        estimator=model,
        search_spaces=param_space,
        scoring="roc_auc",
        cv=inner_cv,
        n_iter=n_iter,
        n_jobs=n_jobs,
        random_state=random_state,
        refit=True,  # fits final model on all data with best params
    )
    opt.fit(X, y)
    final_model = opt.best_estimator_

    # Step 2: Youden threshold via OOF probabilities on full dataset
    oof_proba = np.zeros(len(y))
    for inner_train_idx, inner_val_idx in inner_cv.split(X, y):
        fold_model = opt.best_estimator_.__class__(**opt.best_params_)
        fold_model.fit(X[inner_train_idx], y[inner_train_idx])
        oof_proba[inner_val_idx] = fold_model.predict_proba(X[inner_val_idx])[:, 1]

    fpr, tpr, thresholds = roc_curve(y, oof_proba)
    best_threshold = float(thresholds[np.argmax(tpr - fpr)])

    return final_model, best_threshold, opt.best_params_


# --- Run nested CV first to get honest performance estimates ---
results = nested_cv_youden(
    X=X.values,
    y=y.values,
    model=model,
    param_space=param_space,
    n_splits_outer=5,
    n_repeats_outer=5,
    n_splits_inner=3,
    n_iter=30
)

print(f"Estimated AUC: {results['roc_auc_mean']:.3f} ± {results['roc_auc_std']:.3f}")


#### Train final model on ALL data

In [ ]:
# --- Then train final model on ALL data ---
final_model, final_threshold, best_params = train_final_model(
    X=X.values, y=y.values, model=model, param_space=param_space
)

print(final_model, final_threshold) 
best_params

### Final Model Prediction

In [ ]:
def predict(X_new, model, threshold):
    proba = model.predict_proba(X_new)[:, 1]
    return (proba >= threshold).astype(int), proba

##### Sample Prediction

In [ ]:
test_size = 20
Xnew = X.iloc[:test_size].values
ypred, ypredproba = predict(Xnew, final_model, final_threshold)
ynew=y.iloc[:test_size].values

cm = confusion_matrix(ynew, ypred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
youden_test = (
    sensitivity + specificity - 1
    if not (np.isnan(sensitivity) or np.isnan(specificity))
    else np.nan
)
roc_auc = (
    roc_auc_score(ypred, ypredproba)
    if len(np.unique(ynew)) > 1
    else np.nan
)

print(youden_test, roc_auc)

---

## Optuna Bayes Search Optimization

In [12]:
import numpy as np
from sklearn.datasets import make_classification
from catboost import CatBoostClassifier

In [26]:
model_class= {
    'catboost': CatBoostClassifier,
    'random_forest' : RandomForestClassifier,
}

In [27]:
config.param_space

namespace(iterations=namespace(min=100, max=500),
          depth=namespace(min=4, max=10),
          learning_rate=namespace(min=0.01, max=0.1),
          l2_leaf_reg=namespace(min=1, max=9))

In [22]:
def param_space_fn(trial):
    return {
        "iterations": trial.suggest_int(
            "iterations", config.iterations.min, config.iterations.max),
        "depth": trial.suggest_int(
            "depth", config.depth.min, config.depth.max),
        "learning_rate": trial.suggest_float(
            "learning_rate", config.learning_rate.min, config.learning_rate.max, log=True),
        "l2_leaf_reg": trial.suggest_int(
            "l2_leaf_reg", config.l2_leaf_reg.min, config.l2_leaf_reg.max),
    }

In [28]:
model_class[config.model.name]

catboost.core.CatBoostClassifier

In [23]:
config.optimization

namespace(scoring='roc_auc',
          k_splits_outer=3,
          n_repeats_outer=2,
          k_splits_inner=3,
          n_iter=5)

In [ ]:
results = nested_cv_youden_optuna(
    X=X.values,
    y=y.values,
    model_class=model_class[config.model.name],   # class, not an instance
    param_space_fn=param_space_fn,
    n_splits_outer=config.optimization.k_splits_outer,
    n_repeats_outer=config.optimization.k_repeats_outer,
    n_splits_inner=config.optimization.k_splits_inner,
    n_iter=config.optimization.n_iter,   # Optuna trials per outer fold
    random_state=config.experiment.random_seed,
)

### Calculate Confidence Interval 

In [15]:
import numpy as np

def mean_confidence_interval(results, metric, confidence=0.95):
    scores = [fold[metric] for fold in results['folds']]
    
    scores = np.array(scores)
    n = len(scores)
    mean = np.mean(scores)
    std = np.std(scores, ddof=1)
    stderr = std / np.sqrt(n)

    z = 1.96  # 95% normal approx
    margin = z * stderr

    return {
        "mean": mean,
        "std": std,
        "ci_lower": mean - margin,
        "ci_upper": mean + margin,
        "n_folds": n
    }

In [16]:
youden_ci = mean_confidence_interval(results, "youden")
roc_ci = mean_confidence_interval(results, "roc_auc")

print("Youden 95% CI:", youden_ci)
print("ROC-AUC 95% CI:", roc_ci)

Youden 95% CI: {'mean': 0.8243481324876674, 'std': 0.054742584274136384, 'ci_lower': 0.7885829774285649, 'ci_upper': 0.8601132875467699, 'n_folds': 9}
ROC-AUC 95% CI: {'mean': 0.9692212825933757, 'std': 0.026579751809320113, 'ci_lower': 0.9518558447446199, 'ci_upper': 0.9865867204421315, 'n_folds': 9}


 -----

## Saving the Optimized Model

In [ ]:
def save_optimized_results(name, best_params, best_score, optimized_model, optimized_model_metrics):
    model_results = { 
        "name" : name,
        "best_params": best_params, 
        "best_score": best_score, 
        "optimized_model": optimized_model, 
        "optimized_model_metrics": optimized_model_metrics
    }
    joblib.dump(model_results, rf"outputs\all_features\{name}.pkl")

save_optimized_results("random_forest", opt.best_params_, opt.best_score_, best_model, results)